In [1]:
pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 4.0 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import fitz
import re
import json

PDF_PATH = "/home/tawil/Bureau/stage d'ete/dataset/CODE-PENAL_FR.pdf"
OUTPUT_JSON = "/home/tawil/Bureau/stage d'ete/dataset/code_penal_clean.json"

START_PAGE = 10      # page 11
END_PAGE = 15      # page 234 incluse

def clean_line(line):
    line = line.strip()

    # supprimer numéros de page
    if re.match(r"^-?\s*\d+\s*-?$", line):
        return ""

    # supprimer notes de bas de page
    if re.match(r"^\d+\s*-\s*", line):
        return ""

    # supprimer références courtes
    if line.lower().startswith(("cf.", "ibid")):
        return ""

    # supprimer titres de plage comme "(Articles 13 à 109)"
    if re.match(r"^\(?Articles?\s+\d+\s+à\s+\d+\)?$", line, re.IGNORECASE):
        return ""

    return line

def extract_clean_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_index in range(START_PAGE, END_PAGE):
        page = doc[page_index]
        page_dict = page.get_text("dict")

        # calculer la taille moyenne des polices
        sizes = []
        for block in page_dict["blocks"]:
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    if span["text"].strip():
                        sizes.append(span["size"])

        if not sizes:
            continue

        avg_size = sum(sizes) / len(sizes)

        page_lines = []

        for block in page_dict["blocks"]:
            for line in block.get("lines", []):
                line_text = ""

                for span in line.get("spans", []):
                    text = span["text"]

                    # supprimer petites polices = souvent notes de bas de page
                    if span["size"] < avg_size * 0.85:
                        continue

                    line_text += text

                line_text = clean_line(line_text)

                if line_text:
                    page_lines.append(line_text)

        full_text += "\n".join(page_lines) + "\n"

    return full_text

def remove_remaining_notes(text):
    # supprimer blocs de notes restants
    text = re.sub(
        r"\n\d+\s*-\s*(Article|Alinéa|Les articles|Voir|cf\.|Ibid).*?(?=\nArticle|\nLIVRE|\nTITRE|\nCHAPITRE|\Z)",
        "\n",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # espaces
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_articles(text):
    # Article premier + Article 2 + Article 3...
    pattern = r"(?=\bArticle\s+(?:premier|\d+(?:-\d+)?)\b)"
    parts = re.split(pattern, text, flags=re.IGNORECASE)

    articles = []

    for part in parts:
        part = part.strip()
        if not part.lower().startswith("article"):
            continue

        match = re.match(r"Article\s+(premier|\d+(?:-\d+)?)", part, re.IGNORECASE)
        if not match:
            continue

        number = match.group(1)

        if number.lower() == "premier":
            article_id = "1"
            article_label = "Article premier"
        else:
            article_id = number
            article_label = f"Article {number}"

        articles.append({
            "id": f"penal_article_{article_id}",
            "domain": "droit_penal",
            "source": "Code pénal marocain",
            "article": article_label,
            "text": part
        })

    return articles

def main():
    text = extract_clean_text(PDF_PATH)
    text = remove_remaining_notes(text)

    articles = split_articles(text)

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print("Extraction terminée")
    print("Nombre d'articles :", len(articles))
    print("Premier article :", articles[0]["article"] if articles else "Aucun")
    print("Fichier créé :", OUTPUT_JSON)

if __name__ == "__main__":
    main()

Extraction terminée
Nombre d'articles : 21
Premier article : Article premier
Fichier créé : /home/tawil/Bureau/stage d'ete/dataset/code_penal_clean.json
